# 心语倾诉助理 · StudentConfideAgent

## 项目简介
面向学业压力较大的学生，提供一个合法合规的倾诉 Bot。每位使用者用独立的 `user_id` 沉淀专属记忆。本助理不是心理医生，不做诊断、不开药；出现危机表述时只做转介。

## 作者信息
- GitHub：[@neil43lk](https://github.com/neil43lk)
- 日期：2026-09-29

## 第1部分：环境配置

请在项目目录准备 `.env`（可由 `.env.example` 复制）。没有密钥时，下面的安全检查、记忆隔离和会话小结仍然可以运行。

In [ ]:
import sys
from pathlib import Path

project_name = "neil43lk-StudentConfideAgent"
candidates = [
    Path.cwd(),
    Path.cwd() / "Co-creation-projects" / project_name,
    Path.cwd().parent / project_name,
]
PROJECT_ROOT = next(
    (path.resolve() for path in candidates if (path / "src" / "agents" / "confide_agent.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("未找到项目目录，请从项目目录或仓库根目录启动 Notebook")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.agents.confide_agent import create_confide_agent, replay_samples
from src.safety.guardrails import check_message
from src.utils.summary import render_session_summary

print(f"环境已就绪：{PROJECT_ROOT}")

## 第2部分：安全护栏

危机、诊断开药、违法和色情请求在进入大模型之前就被拦住。危机场景只返回转介话术。

In [ ]:
samples = [
    "最近考试压力好大，又不知道怎么跟爸妈说。",
    "我真的不想活了",
    "你能给我开点抗焦虑的药吗",
]
for text in samples:
    result = check_message(text)
    print("=" * 40)
    print(f"输入：{text}")
    print(f"类别：{result.category}，放行：{result.allowed}")
    if result.message:
        print(result.message)


## 第3部分：专属记忆

回放虚构示例，并重置 `student_xiaoming`、`student_xiaohong`、`safety_demo` 三个示例档案，避免重复运行把次数叠上去。小明和小红的主题必须互相看不见。未配置 API 密钥时，助理会说明无法生成倾听回复，但主题仍会记下。

In [ ]:
demo = replay_samples()
safety_rows = [item for item in demo["transcripts"] if item["kind"] == "safety"]
print("安全用例：")
for row in safety_rows:
    print(f"- 期望 {row['expected']}，实际 {row['category']}，通过 {row['passed']}")

print("\n专属主题：")
for user_id, profile in demo["profiles"].items():
    if user_id == "safety_demo":
        print(f"- {user_id} 拦截计数：{profile.get('blocked', {})}")
        continue
    print(f"- {user_id}：{profile.get('themes', {})}")


## 第4部分：单用户对话

与 README 中的调用方式一致。有密钥时这里会请求大模型；没有密钥时只确认记忆写入。

In [ ]:
agent = create_confide_agent(user_id="student_xiaoming")
reply = agent.chat("我还是不知道明天怎么跟爸妈开口。")
print(reply)
print("\n当前主题：", agent.profile().get("themes"))


## 第5部分：会话小结

小结只统计通过安全检查的倾诉，不把危机原文写进报告。

In [ ]:
summary_path = agent.write_summary()
print(render_session_summary(agent.profile()))
print(f"\n已写入：{summary_path}")


## 第6部分：总结

### 已实现
- 倾诉入口：`SimpleAgent`，系统提示约束共情与合规边界
- 进模型前的安全护栏，危机场景转介 12356 / 120 / 110
- 按 `user_id` 隔离的本地专属记忆；配置 LLM 后附加 HelloAgents 工作记忆

### 边界
本 Notebook 使用虚构例句。它不能替代学校心理老师、医院或危机干预。

### 接下来
- 补一组人工评测，看共情质量和越界率
- 评测集扩大后，再考虑极简聊天页